# UEBA Portable — **Évaluation de la détection** (T1110.003)

Mesure la capacité du modèle à **détecter une attaque connue**, en respectant
la méthodologie d'évaluation d'un détecteur non supervisé :

1. on **apprend** la baseline sur les **jours propres** (attaque exclue) ;
2. on **score l'intégralité** des données (jours propres **+** jours d'attaque) ;
3. on compare aux étiquettes connues → **Recall, Précision, F1, FP, matrice de
   confusion**, et le **Recall opérationnel** (≥ 1 alerte par utilisateur × jour).

> Charge ici ton CSV **complet** (tous les jours, attaque incluse).

## 0. Installation

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/assia-xnz/ueba-portable.git"
    print("✅ Installé. Si des cellules ont déjà tourné : Exécution → Redémarrer la session.")
else:
    sys.path.insert(0, "../src")
print("Environnement :", "Google Colab" if IN_COLAB else "Local")

## 1. Upload du CSV complet (tous les jours, attaque incluse)

In [ ]:
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))
else:
    CSV_PATH = "../tests/integration/fixtures/sample_logs.csv"
print("Dataset :", CSV_PATH)

## 2. Vérité terrain de l'attaque

Renseigne les jours et les comptes ciblés par l'attaque connue dans **ce**
fichier.

In [ ]:
ATTACK_DATES = ["2026-05-13", "2026-05-16"]
ATTACK_USERS = {
    "a.amrani",
    "l.idrissi",
    "l.mus",
    "y.ben",
    "n.alam",
    "s.ed",
    "k.alaa",
}
print("Jours d'attaque :", ATTACK_DATES)
print("Comptes ciblés  :", sorted(ATTACK_USERS))

## 3. Normalisation et extraction des features

In [ ]:
import csv
from datetime import timedelta
from pathlib import Path

from ueba.adapters.wazuh import WazuhAdapter
from ueba.domain.features import UEBAFeatureExtractor

with Path(CSV_PATH).open(newline="", encoding="utf-8") as f:
    records = list(csv.DictReader(f))
events = WazuhAdapter().normalize(records)

extractor = UEBAFeatureExtractor(
    window_size=timedelta(hours=1), window_step=timedelta(minutes=30)
)
vectors = extractor.extract(events)

dates = sorted({v.window_start.date().isoformat() for v in vectors})
print(f"Événements : {len(events)} | Fenêtres : {len(vectors)}")
print(f"Période    : {dates[0]} → {dates[-1]} ({len(dates)} jours)")

## 4. Entraînement sur les jours PROPRES (attaque exclue du train)

C'est la séparation méthodologique : la baseline n'apprend **que** du normal.

In [ ]:
from ueba.domain.per_user_ensemble import PerUserAnomalyEnsemble

MIN_WINDOWS = 30  # à calibrer (baisse si peu de fenêtres par utilisateur)

train_vectors = [v for v in vectors if v.window_start.date().isoformat() not in ATTACK_DATES]
print(f"Fenêtres d'apprentissage (propres) : {len(train_vectors)} / {len(vectors)}")

model = PerUserAnomalyEnsemble(
    min_windows_per_user=MIN_WINDOWS,
    train_ratio=1.0,
    svm_nu=0.05,
    n_estimators=200,
    majority_threshold=2,
    random_state=42,
)
model.fit(train_vectors)
print(f"Modèles entraînés : {len(model.trained_users)}")

## 5. Détection sur tout + métriques

On score **toutes** les fenêtres et on compare aux étiquettes. Une *fenêtre
d'attaque* = (utilisateur ciblé) ET (jour d'attaque).

In [ ]:
import pandas as pd

verdicts = model.predict(vectors)
df = pd.DataFrame(
    {
        "user": [v.user for v in vectors],
        "date": [v.window_start.date().isoformat() for v in vectors],
        "window_start": [v.window_start for v in vectors],
        "is_anomaly": [d.is_anomaly for d in verdicts],
        "was_in_training": [d.was_in_training for d in verdicts],
    }
)
df["is_attack"] = df["user"].isin(ATTACK_USERS) & df["date"].isin(ATTACK_DATES)

# Couverture : des comptes ciblés sans modèle fausseraient le FP (default-deny).
untrained_non_attack = df[(~df["is_attack"]) & (~df["was_in_training"])]["user"].nunique()
if untrained_non_attack:
    print(f"⚠️  {untrained_non_attack} utilisateur(s) normal(aux) sans modèle "
          f"(default-deny) → baisse MIN_WINDOWS pour un FP juste.\n")

# Matrice de confusion (niveau fenêtre)
TP = int((df["is_attack"] & df["is_anomaly"]).sum())
FN = int((df["is_attack"] & ~df["is_anomaly"]).sum())
FP = int((~df["is_attack"] & df["is_anomaly"]).sum())
TN = int((~df["is_attack"] & ~df["is_anomaly"]).sum())

recall = TP / (TP + FN) if (TP + FN) else float("nan")
precision = TP / (TP + FP) if (TP + FP) else float("nan")
f1 = 2 * precision * recall / (precision + recall) if precision and recall else float("nan")
fp_rate = FP / (FP + TN) if (FP + TN) else float("nan")

# Recall opérationnel : (user × jour) d'attaque avec au moins une alerte
attack = df[df["is_attack"]]
op = attack.groupby(["user", "date"])["is_anomaly"].any()
op_recall = op.mean() if len(op) else float("nan")
first = attack.sort_values("window_start").groupby(["user", "date"]).first()["is_anomaly"]

print("===== MATRICE DE CONFUSION (fenêtres) =====")
print(f"  TP={TP:5d}   FN={FN:5d}")
print(f"  FP={FP:5d}   TN={TN:5d}\n")
print("===== MÉTRIQUES =====")
print(f"  Recall (fenêtre)        : {recall:6.1%}")
print(f"  Précision               : {precision:6.1%}")
print(f"  F1-score                : {f1:6.1%}")
print(f"  FP rate                 : {fp_rate:6.1%}")
print(f"  Recall opérationnel     : {op_recall:6.1%}  (user×jour avec ≥1 alerte)")
print(f"  Détection 1re fenêtre   : {int(first.sum())}/{len(first)}")

## 6. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (axc, axt) = plt.subplots(1, 2, figsize=(14, 5))

# (a) Matrice de confusion
mat = np.array([[TP, FN], [FP, TN]])
im = axc.imshow(mat, cmap="Blues")
axc.set_xticks([0, 1]); axc.set_xticklabels(["prédit\nanomalie", "prédit\nnormal"])
axc.set_yticks([0, 1]); axc.set_yticklabels(["réel\nattaque", "réel\nnormal"])
for i in range(2):
    for j in range(2):
        axc.text(j, i, mat[i, j], ha="center", va="center",
                 color="white" if mat[i, j] > mat.max() / 2 else "black", fontsize=14)
axc.set_title("Matrice de confusion (fenêtres)")

# (b) Timeline des détections
normal = df[~df["is_anomaly"]]
flagged = df[df["is_anomaly"]]
axt.scatter(normal["window_start"], normal["user"], s=10, c="#aec7e8", label="normal")
axt.scatter(flagged["window_start"], flagged["user"], s=22, c="#d62728", label="alerte")
for d in ATTACK_DATES:
    axt.axvspan(np.datetime64(f"{d}T00:00:00"), np.datetime64(f"{d}T23:59:59"),
                color="orange", alpha=0.12)
axt.set_title("Timeline (zones orange = jours d'attaque)")
axt.legend(loc="upper right"); axt.tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()

## 7. Lecture des résultats

* **Recall opérationnel élevé** (idéalement 100 %) = le SOC est alerté au moins
  une fois pour chaque utilisateur ciblé chaque jour d'attaque — c'est la
  métrique de référence en littérature SOC (Bhatt et al. 2014).
* **FP rate bas** = peu de bruit sur le normal (réglable via `svm_nu`,
  `majority_threshold`, `reconstruction_error_percentile`).
* Le compromis Recall ↔ FP se pilote par ces hyperparamètres : documente-le
  dans ton rapport (courbe FP vs Recall pour quelques valeurs de `svm_nu`).